# Building GPT

In [135]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [136]:
# getting the input.

words = open("input.txt").read()

In [137]:
# now, we build the vocabulary.
# vocabulary is nothing but how many unique characters are there in the input text. we will use this to encode the text into integers.

vocab = sorted(list(set(words)))
vocab_size = len(vocab)
print(''.join(vocab))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [138]:
print(vocab_size)

65


In [139]:
# we're creating a mapping from char to int, and vice-versa.

stoi = { ch:i for i,ch in enumerate(vocab) }
itos = { i:ch for i,ch in enumerate(vocab) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a strin

## Making our dataset now.

### We're going to be splitting this into train and val sets.
### train - 90%, val - 10%.

In [140]:
data = torch.tensor(encode(words), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [141]:
# data[:1000]

In [142]:
# now that we have the data in the form of a tensor
# we want to split this into train and val sets.

n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

In [143]:
# get_batch basically just provides a single batch of inputs to the network.
# it creates the inputs, and also the targets against which the inputs will be evaluated (loss).

torch.manual_seed(1337)
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embed = 64

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

# print('----')

# for b in range(batch_size): # batch dimension
#     for t in range(block_size): # time dimension
#         context = xb[b, :t+1]
#         target = yb[b,t]
#         print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([16, 32])
tensor([[58, 53,  1, 41, 53, 56, 56, 59, 54, 58,  1, 39,  1, 51, 39, 52,  5, 57,
          1, 61, 47, 44, 43,  1, 47, 57,  0, 61, 46, 43, 52,  1],
        [49,  1, 39, 52,  1, 53, 39, 58, 46,  1, 40, 63,  1, 20, 47, 51,  6,  0,
         32, 46, 43,  1, 59, 52, 47, 58, 63,  1, 58, 46, 43,  1],
        [59, 50, 42,  1, 58, 46, 53, 59,  1, 61, 43, 56, 58,  1, 57, 53,  1, 58,
         53, 53,  2,  0,  0, 24, 33, 15, 21, 27, 10,  0, 35, 43],
        [ 8,  0,  0, 35, 13, 30, 35, 21, 15, 23, 10,  0, 28, 56, 53, 60, 43,  1,
         47, 58,  6,  1, 20, 43, 52, 56, 63,  6,  1, 39, 52, 42],
        [58,  1, 57, 46, 43,  8,  0,  0, 32, 30, 13, 26, 21, 27, 10,  0, 18, 53,
         56,  1, 61, 46, 39, 58,  1, 56, 43, 39, 57, 53, 52,  6],
        [56, 61, 47, 41, 49,  6,  1, 50, 43, 58,  1, 47, 58,  1, 40, 43, 11,  0,
         18, 53, 56,  1, 47, 52,  1, 58, 46, 63,  1, 57, 46, 53],
        [25, 10,  0, 35, 47, 58, 46, 42, 56, 39, 61,  1, 63, 53, 59,  1, 46, 43,
        

In [ ]:
# now, we're going to build up an embedding lookup table.
# this is basically a giant table that contains the embedding vectors for each character in the vocab.
# so, this is going to be of shape (65, 32).
# 65 because we have 65 unique chars in the vocab, and 32 as the embedding dimension because we want each char to be represented as a 32-dimensional vector.

# we also want to add positional embeddings, which will help the model understand the order of the characters in the input sequence.
# because, the order matters too. "cat in a hat" is different from "hat in a cat".
# and how are these gotten and represented?
# similar to the token embeddings, we can have a posi embedding block of tunable parameters, of shape (block_size, n_embed).
# so, logits (without batches) are going to be (T, n_embed), and the posi embeddings are going to be (T, n_embed) as well, so we can just add them together.


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # (65, 32)
        self.position_embedding_table = nn.Embedding(block_size, n_embed) # (8, 32)
        self.lm_head = nn.Linear(n_embed, vocab_size) # (32, 65)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeds = self.token_embedding_table(idx) # (B, T, C)
        position_embeds = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        X = token_embeds + position_embeds
        logits = self.lm_head(X) # (B, T, vocab_size)

        # now, we want to do a forward pass, which is just calculating the logits, and the loss
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [145]:
m = BigramLanguageModel()

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [146]:
# the initial loss, before any backprop even happens is 4.6695.
# now, let's automate the forward & backward passes.


batch_size = 32

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    print(f"Step {steps}, Loss: {loss.item()}")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 0, Loss: 4.494860649108887
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 1, Loss: 4.431762218475342
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 2, Loss: 4.42417573928833
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 3, Loss: 4.430331230163574
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 4, Loss: 4.439373016357422
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 5, Loss: 4.371947288513184
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
Step 6, Loss: 4.346046447753906
token_embeds shape is:  torch.Size([32, 32, 64])
position_embeds shape is:  torch.Size([32, 64])
S

In [147]:
# now, how're we going to generate output from the trained model?
# we have to get the output's, i.e. the softmax probabilities, and keep picking the most probable out of them at every turn
# we have to rememeber that we're not training here, just generating. so, no backprop.

context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


token_embeds shape is:  torch.Size([1, 1, 64])
position_embeds shape is:  torch.Size([1, 64])
token_embeds shape is:  torch.Size([1, 2, 64])
position_embeds shape is:  torch.Size([2, 64])
token_embeds shape is:  torch.Size([1, 3, 64])
position_embeds shape is:  torch.Size([3, 64])
token_embeds shape is:  torch.Size([1, 4, 64])
position_embeds shape is:  torch.Size([4, 64])
token_embeds shape is:  torch.Size([1, 5, 64])
position_embeds shape is:  torch.Size([5, 64])
token_embeds shape is:  torch.Size([1, 6, 64])
position_embeds shape is:  torch.Size([6, 64])
token_embeds shape is:  torch.Size([1, 7, 64])
position_embeds shape is:  torch.Size([7, 64])
token_embeds shape is:  torch.Size([1, 8, 64])
position_embeds shape is:  torch.Size([8, 64])
token_embeds shape is:  torch.Size([1, 9, 64])
position_embeds shape is:  torch.Size([9, 64])
token_embeds shape is:  torch.Size([1, 10, 64])
position_embeds shape is:  torch.Size([10, 64])
token_embeds shape is:  torch.Size([1, 11, 64])
position_e